In [ ]:
import numpy as np
from h5py import File
import matplotlib.pyplot as plt
import fish_proc.wholeBrainDask.cellProcessing_single_WS as fwc
import dask.array as da
import os
import pandas as pd
import seaborn as sns
sns.set(font_scale=1.5, style='ticks')
df = pd.read_csv('datalist.csv', index_col=0)

def cell_loc(cell_id):
    thres_ = 0.005
    w_ = (W[cell_id]>thres_)*W[cell_id]
    w_[np.isnan(w_)]=0
    x_loc, y_loc, z_loc = X[cell_id],Y[cell_id],Z[cell_id]
    return (z_loc.dot(w_))/w_.sum(), (x_loc.dot(w_))/w_.sum(), (y_loc.dot(w_))/w_.sum()

In [ ]:
ind = 8
row = df.iloc[ind]
dir_ = row['dir_']+'/seg_mika/'
save_root = row['save_root']
if not os.path.exists(save_root):
    os.makedirs(save_root)

In [ ]:
cell_file = File(dir_+'cells0_clean.hdf5', 'r')
volume_file = File(dir_+'volume0.hdf5', 'r')
cell_file.keys(), volume_file.keys()

In [ ]:
X=cell_file['cell_x']
Y=cell_file['cell_y']
Z=cell_file['cell_z']
W=cell_file['cell_weights'][()]
V=cell_file['volume_weight']

F = cell_file['cell_timeseries_raw'][()]
background = cell_file['background'][()]
brain_map=volume_file['volume_mean'][()]

# remove background from raw F
F = F - (background - 10)
F[F<0]=0
F_dask = da.from_array(F, chunks=('auto', -1))
win_ = 400
baseline_ = da.map_blocks(fwc.baseline, F_dask, dtype='float', window=win_, percentile=20, downsample=10).compute()
dFF = F/baseline_-1

In [ ]:
# plt.plot(F.mean(axis=0), '-k', lw=2)
# plt.plot(baseline_.mean(axis=0), '-r', lw=2)
# plt.ylabel('F')
# plt.xlabel('Time (s)')
# sns.despine()
# plt.show()

In [ ]:
brain_shape = V.shape
np.savez(save_root+'cell_dff.npz', \
         dFF=dFF.astype('float16'), \
         baseline=baseline_.astype('float16'), \
         brain_shape=brain_shape, \
         X=X, Y=Y, Z=Z, W=W)
np.save(save_root+'Y_ave.npy', brain_map)

numCells = F.shape[0]
A_center = np.zeros((numCells,3))
for n_cell in range(numCells):
    A_center[n_cell] = cell_loc(n_cell)
np.save(save_root+'cell_center.npy', A_center)

### brain map

In [ ]:
save_root

In [ ]:
brain_map = np.load(save_root+'Y_ave.npy')

In [ ]:
plt.figure(figsize=(10, 8))
plt.imshow(brain_map.max(axis=0), vmin=100)
# plt.colorbar()
plt.show()

### Cell locations

In [ ]:
A_center = np.load(save_root+'cell_center.npy')

In [ ]:
plt.scatter(A_center[:,1], A_center[:,2], s=1)
plt.show()